# 字节串(`bytes`)的构造

In [ ]:
import io; 
from struct import Struct; 

In [ ]:
#显示bytes对象的十六进制内容
def bytes_hexview(b: bytes): 
    strm_bin = io.BytesIO(b); 
    strm_hex = io.StringIO(b.hex()); 
    #每行显示8个字节
    while True: 
        #显示下一个字节相对于开头的偏移量
        print("{0:0>16x}".format(strm_bin.tell()), end="\x20" * 2); 
        ch_byte = strm_bin.read(8); 
        len_pedding = 8 - len(ch_byte); 
        #打印每个字节的十六进制码, 相邻字节间以空格分割
        for _ in range(8): 
            print(strm_hex.read(2), end="\x20"); 
        print("\x20" * (len_pedding * 2 + 1), end=str())
        #输出8个字节对应的字符
        #其中, 0x20至0x7e输出为对应的ASCII可打印字符, 其他字节输出为半角句点
        for ch in ch_byte: 
            if 32 <= ch <= 126: 
                print(chr(ch), end=str()); 
            else: 
                print(".", end=str()); 
        print("\n", end=str()); 
        if len_pedding > 0: 
            break; 

## `bytes`对象的显式构造
`bytes`是一种与`str`相似的对象, 在数据结构中同属于"串". 但`bytes`中每个元素都是字长固定为8位 (1字节), 取值介于`0`(含)~`255`(含)之间的整数. 

`bytes`的声明方法与`str`相似, 使用一对**半角**双引号(或一对**半角**单引号)括注若干字符, 但**需在定界符之前插入`b`**. 此外, `bytes`的定界符之内接受的字符类型存在限制, 只能包含以下字符: 

* 使用转义规则`\x`后接两位十六进制数码所表示的字符; 
* `\a`, `\b`, `\t`, `\n`, `\v`, `\f`, `\r` (即`\x07`至`\x0d`); 
* `ASCII`可打印字符 (即`\x20`至`\x7e`), 但`"`, `'`和`\`需分别转义为`\"`, `\'`, `\\` (与`str`中的转义规则相同). 

In [ ]:
motto_la = u"Verba volant, scripta manent"; 
motto_la_bin = b"Verba volant, scripta manent"; 
print(type(motto_la), type(motto_la_bin)); 

## `bytes`对象的隐式构造

`bytes`对象的隐式构造方式包括: 
* `bytes`对象的拼接; 
* 采用特定编码规则, 对`str`对象中的字符内容进行编码; 
* 使用其他迭代器构造, 要求其每次调用`__next__`方法的返回结果均可被转换为`int`; 
* 使用`list`, `tuple`等内建可迭代对象构造, 要求其每个元素均为`int`; 
* 使用`bytes`类的`fromhex`方法; 
* 使用`struct.Struct`对象的`pack`方法; 

### `bytes`对象的拼接
支持`__add__`, `__mul__`和`join`方法, 功能和用法与`str`对象对应的同名方法相同, 但要求参与拼接的所有序列均为`bytes`, 不支持`str`与`bytes`的混合拼接. 

In [4]:
b"A" + b"B" * 3 + b"C"

b'ABBBC'

In [5]:
b" < ".join([b"1", b"2", b"3"])

b'1 < 2 < 3'

### `str`对象的编码与`bytes`对象的解码

|方法|功能|备注|
|:-|:-:|:-|
|`text.encode(rule)`|采用`rule`对`text`逐字符编码, <br>得到按字符顺序无重复无间隔<br>排列的二进制序列|`text`为`str`对象, <br>方法的返回结果为`bytes`对象|
|`binary.decode(rule)`|采用`rule`对`binary`解码|`binary`为`bytes`对象, <br>方法的返回结果为`str`对象|

对比基本拉丁字母和汉字分别在`UTF-8`, `UTF-16`和`UTF-32`编码下的差异

In [6]:
enc_rules = tuple("utf-{len:d}".format(len=l) for l in (8, 16, 32)); 

In [7]:
for rule in enc_rules: 
    print(rule); 
    bytes_hexview(motto_la.encode(rule)); 

utf-8
0000000000000000  56 65 72 62 61 20 76 6f  Verba vo
0000000000000008  6c 61 6e 74 2c 20 73 63  lant, sc
0000000000000010  72 69 70 74 61 20 6d 61  ripta ma
0000000000000018  6e 65 6e 74              nent
utf-16
0000000000000000  ff fe 56 00 65 00 72 00  ..V.e.r.
0000000000000008  62 00 61 00 20 00 76 00  b.a. .v.
0000000000000010  6f 00 6c 00 61 00 6e 00  o.l.a.n.
0000000000000018  74 00 2c 00 20 00 73 00  t.,. .s.
0000000000000020  63 00 72 00 69 00 70 00  c.r.i.p.
0000000000000028  74 00 61 00 20 00 6d 00  t.a. .m.
0000000000000030  61 00 6e 00 65 00 6e 00  a.n.e.n.
0000000000000038  74 00                    t.
utf-32
0000000000000000  ff fe 00 00 56 00 00 00  ....V...
0000000000000008  65 00 00 00 72 00 00 00  e...r...
0000000000000010  62 00 00 00 61 00 00 00  b...a...
0000000000000018  20 00 00 00 76 00 00 00   ...v...
0000000000000020  6f 00 00 00 6c 00 00 00  o...l...
0000000000000028  61 00 00 00 6e 00 00 00  a...n...
0000000000000030  74 00 00 00 2c 00 00 00  t...,...
00

In [8]:
motto_zh = u"岁月失语，唯石能言"; 
for rule in enc_rules: 
    print(rule); 
    bytes_hexview(motto_zh.encode(rule)); 

utf-8
0000000000000000  e5 b2 81 e6 9c 88 e5 a4  ........
0000000000000008  b1 e8 af ad ef bc 8c e5  ........
0000000000000010  94 af e7 9f b3 e8 83 bd  ........
0000000000000018  e8 a8 80                 ...
utf-16
0000000000000000  ff fe 81 5c 08 67 31 59  ...\.g1Y
0000000000000008  ed 8b 0c ff 2f 55 f3 77  ..../U.w
0000000000000010  fd 80 00 8a              ....
utf-32
0000000000000000  ff fe 00 00 81 5c 00 00  .....\..
0000000000000008  08 67 00 00 31 59 00 00  .g..1Y..
0000000000000010  ed 8b 00 00 0c ff 00 00  ........
0000000000000018  2f 55 00 00 f3 77 00 00  /U...w..
0000000000000020  fd 80 00 00 00 8a 00 00  ........
0000000000000028                           


### 使用仅包含可转换为`int`类元素的迭代器构造
用法: 
```python
bytes(iter_0)
```

* `iter_0` 迭代器 (或其子类) 的实例, 
    * **每次调用`__next__()`方法**返回的结果均**可被转换为`int`**; 
    * `0 <= next(iter_0) <= 255`

In [9]:
#构造生成器用于逐个生成int, 它是迭代器的子类, 继承了__next__方法. 
code_ascii = (ord(ch) for ch in motto_la); 
print(code_ascii); 
for _ in range(6): 
    print(next(code_ascii)); 
print(bytes(code_ascii)); 

<generator object <genexpr> at 0x000000000526E7C8>
86
101
114
98
97
32
b'volant, scripta manent'


### 使用部分内建可迭代对象构造
用法: 
```python
bytes(iter_0)
```

* `iter_0` 可迭代对象的实例
    * 支持的可迭代对象类型: `list`, `tuple`, `set`等
    * **每个元素均可被转换为`int`**
    * `0 <= int(elem) <= 255`

In [10]:
print(bytes([1, 2, 3])); 

b'\x01\x02\x03'


### 使用`bytes.fromhex`方法构造

用法: 
```python
bytes.fromhex(h)
```

* `h` 用于构造`bytes`的`str`, 
* `h`只能包含以下字符: `0`\~`9`, `A`\~`F`, `a`\~`f`, 
    以及下列空白字符: `\t`, `\n`, `\v`, `\f`, `\r`, `\x20` (半角空格); 
* 在`bytes`对象的构造过程中, 不区分字母大小写; 
* `h`中包含的**非空白字符**数量必须为偶数; 
* `h`中的**非空白字符**将以**连续的两个字符**为一组, 无重叠, 无间隔地将其表示的十六进制数解析为对应的字节; \
    **同一组的两个字符必须相邻**; 不同组的字符间可以相邻, 或者插入任意个数的上述空白字符; \
    返回的`bytes`中各个字节的顺序与构造过程中各组出现的顺序相同. 

In [11]:
print(bytes.fromhex("Aced decade")); 

b'\xac\xed\xde\xca\xde'


### 构建`struct.Struct`对象, 并调用其`pack`方法构造`bytes`

#### `struct.Struct`对象的构造
用法: 
```python
struct.Struct(bytes_template)
```

* `bytes_template` 用于构造`bytes`的模板, `str`对象, 

`bytes_template` 的内容包含两部分: 

`order` `item`

* `order` 字节的顺序

|选项|作用|
|:-:|:-|
|`@`|当前解释器默认采用的字节顺序|
|(省略)|同`@`|
|`<`|小端式, 数据的低字节存放在RAM的低地址|
|`>`或`!`|大端式, 数据的高字节存放在RAM的低地址|

* `item` 待生成或解析的二进制序列的结构
    * `item`可以包含以下字符: 
        * 表示特定数据类型的字母: 

            ||8位<br>(1字节)|16位<br>(2字节)|32位<br>(4字节)|64位<br>(8字节)|备注|
            |:-|:-:|:-:|:-:|:-:|:-|
            |占位符|`x`||||在调用`pack`方法时, 将在<br>相应位置的字节处输出`\x00`; <br>当调用`unpack`方法时, 对<br>相应位置的字节不予解析|
            |有符号整型|`b`|`h`|`l`|`q`||
            |无符号整型|`B`|`H`|`L`|`Q`||
            |浮点型||`e`|`f`|`d`|执行`IEEE 754`标准; <br>`float16`在 python 3.6 引入|
            |单字节|`c`||||在调用`pack`方法时, 对应的<br>位置传入的参数必须为`bytes`, <br>且长度为1|
            |字节串|`s`||||在调用`pack`方法时, 对应的<br>位置传入的参数必须为`bytes`|
            
        * 数字`0`~`9`
        * 空白字符: `\t`, `\n`, `\v`, `\f`, `\r`, `\x20` (半角空格)
        
    * 除字节串(`s`)外, 同种数据类型**连续多次**出现时, 可以合并成一个字母, **其前紧邻**该字母出现的次数; \
        如`3d`等效于`ddd`; 
        
    * 字节串`s`前紧邻数字时, 表示待生成或解析的字节串中, 该位置是**单一的**, 特定字长的字节串子串; 
    
    * 字母与其**之前**的数字之间必须相邻; 
    
    * 不同字母间, 或者字母与其**之后**的数字之间可以相邻, 或者插入任意个数的上述空白字符; 

#### `struct.Struct`对象的主要方法

|方法|功能|备注|
|:-|:-:|:-|
|`pack(*data)`|利用`*data`生成字节串|`*data` 用于输出字节串的数据, 在待生成<br>字节串中的顺序与传入至方法中的顺序相同; <br>方法的返回结果为`bytes`对象|
|`unpack(binary)`|从字节串`binary`解析出数据|字节串长度必须与对象的`size`属性值相同; <br>方法的返回结果为`tuple`对象|

In [12]:
#比较不同字长的整型在定长存储时的差异
print(42); 
bin_int_template = tuple(Struct(">" + mode) for mode in "bhlq"); 
for temp in bin_int_template: 
    print(temp.format); 
    bytes_hexview(temp.pack(42)); 

42
>b
0000000000000000  2a                       *
>h
0000000000000000  00 2a                    .*
>l
0000000000000000  00 00 00 2a              ...*
>q
0000000000000000  00 00 00 00 00 00 00 2a  .......*
0000000000000008                           


In [13]:
#比较不同字长的浮点型在定长存储时的差异
import math; 
print(math.pi); 
bin_float_template = tuple(Struct(">" + mode) for mode in "efd"); 
for temp in bin_float_template: 
    print(temp.format); 
    bytes_hexview(temp.pack(math.pi)); 

3.141592653589793
>e
0000000000000000  42 48                    BH
>f
0000000000000000  40 49 0f db              @I..
>d
0000000000000000  40 09 21 fb 54 44 2d 18  @.!.TD-.
0000000000000008                           


In [14]:
#比较c与s的差异
n_bytes = len(motto_la_bin); 
print(motto_la_bin, n_bytes); 
bin_chars_template = tuple(
    Struct(">{n}{type}".format(n=n_bytes, type=mode))
    for mode in "cs"
); 
for temp in bin_chars_template: 
    print(temp.format); 
    print(temp.unpack(motto_la_bin)); 

b'Verba volant, scripta manent' 28
>28c
(b'V', b'e', b'r', b'b', b'a', b' ', b'v', b'o', b'l', b'a', b'n', b't', b',', b' ', b's', b'c', b'r', b'i', b'p', b't', b'a', b' ', b'm', b'a', b'n', b'e', b'n', b't')
>28s
(b'Verba volant, scripta manent',)
